In [ ]:
import numpy as np
import plumed
import matplotlib.pyplot as plt
import os
from scipy import stats
from scipy.spatial.distance import jensenshannon
import seaborn as sns
def get_cvname(filename):
    data = plumed.read_as_pandas(filename)
    cv_name = []
    for i in data.columns[1:]:
        if i.startswith('e') or i.startswith('sin') or i.startswith('cos'):
            pass
        else:
            cv_name.append(i)
    return cv_name

def calculate_distribution(cv_name, colvar,sigma=0.1):
    try:
        os.mkdir("FES")
    except FileExistsError:
        pass
    for cv in cv_name:
        cv_str = ",".join(cv)
        sigmas = [sigma for _ in range(len(cv))]
        sigmas = ",".join(map(str, sigmas))
        print(sigmas)
        os.system(f"rm -rf /workdir/AgOR10_28/AgOR10/PCCA2/FES/FES_{colvar}_{cv_str}.dat")
        os.system(f"python /root/FES_from_Reweighting.py -f /workdir/AgOR10_28/AgOR10/PCCA2/COLVAR_alpha_angle{colvar} -o /workdir/AgOR10_28/AgOR10/PCCA2/FES/FES_alpha_angle{colvar}_{cv_str}.dat --sigma {sigmas} --temp 310 --cv {cv_str} --bin 20 --bias no")

cv_names = [['helix']]
state_indices = [0, 1, 2, 3, 4, 5]
print("Calculating distributions...")
for idx in state_indices:
    calculate_distribution(cv_names, str(idx))

In [ ]:
def plot_all_distributions(cv_name, filenames,indices, output_file=None):
    plt.figure(figsize=(12, 7))
    
    # 使用标准配色方案
    
    cmap = plt.get_cmap('viridis')
    num_states = len(indices)
    # colors = [cmap(i / (num_states-1)) for i in range(num_states)]
    colors = [
    "#F6C667",  # soft gold
    "#B8A1E3",
      "#00F56A",  # light lavender
        "#00B8D4",  # mint-teal
        "#F4CFB6",
        
        ]
    kb = 0.0083145
    T = 310
    kbT = kb * T
    sns.set_theme(style="ticks") # 清爽的刻度样式
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.size': 12,
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5
        })
    # apo_start_file = plumed.read_as_pandas(f"/workdir/AgOR10_28/AgOR10/AgOR10_apo/cuts1/COLVAR_alpha_angle_start")
    # apo_start = apo_start_file[f'{cv_name}'][0]
    # bound_start_file = plumed.read_as_pandas(f"/workdir/AgOR10_28/AgOR10/AgOR10_bound/COLVAR_alpha_angle_start")
    # bound_start = bound_start_file[f'{cv_name}'][0]
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, filename in enumerate(filenames):
        file_path = f"/workdir/AgOR10_28/AgOR10/PCCA2/FES/{filename}"
        index = indices[i]
        if not os.path.exists(file_path):
            continue
            
        data = plumed.read_as_pandas(file_path)
        
        # 1. 计算概率密度
        prob = np.exp(-data['file.free'] / kbT)
        prob = prob / np.sum(prob)*100 # 归一化()

        # 2. 排序以便绘图
        cv_values = data[cv_name].values
        if cv_name == 'angle':
            cv_values = cv_values/np.pi*180

        sort_idx = np.argsort(cv_values)
        ax.plot(cv_values[sort_idx], prob[sort_idx], 
                 label=f'{index}', color=colors[i], lw=2)
        # 可选：填充透明区域，增加视觉效果
        ax.fill_between(cv_values[sort_idx], prob[sort_idx], alpha=0.1, color=colors[i])
    # if cv_name == 'angle':
    #     apo_start = apo_start/np.pi*180
    #     bound_start = bound_start/np.pi*180
    # apo_start = 0.62
    # bound_start = 0.97
    # ax.axvline(x=apo_start, color='black', linestyle='--', linewidth=2, label='Apo-Closed Cryo-EM')
    # ax.axvline(x=bound_start, color='red', linestyle='--', linewidth=2, label='Holo-Open Cryo-EM')
        # 4. 精修图表细节
    ax.set_xlabel(r'ALPHARMSD', labelpad=10)
    # ax.set_xlabel(r'ALPHARMSD', labelpad=10)
    ax.set_ylabel('Probability Density (%)', labelpad=10)
    # ax.set_xlim(0, 0.5)
    # ax.set_ylim(0, 16)
    # 去掉上方和右方的边框
    sns.despine()

    # 添加网格（可选，淡淡的灰色网格有助于读数）
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # 优化图例
    ax.legend(frameon=False, loc='upper right', fontsize=11)

# 自动调整布局，防止边缘裁剪
    plt.tight_layout()
    plt.savefig(f'{cv_name}.pdf', dpi=300, transparent=True)
    plt.show()
plot_all_distributions('helix', ['FES_1_helix.dat','FES_2_helix.dat','FES_3_helix.dat','FES_4_helix.dat'], [
    r"$\mathrm{I'}$",
    r"$\mathrm{I}$",
    r"$\mathrm{O}$",
    r"$\mathrm{C}$",
])

In [ ]:
def plot_all_distributions(cv_name, filenames,indices, output_file=None):
    plt.figure(figsize=(12, 7))
    
    # 使用标准配色方案
    
    cmap = plt.get_cmap('viridis')
    num_states = len(indices)
    # colors = [cmap(i / (num_states-1)) for i in range(num_states)]
    colors = [
    "#F6C667",  # soft gold
    "#B8A1E3",
      "#00F56A",  # light lavender
        "#00B8D4",  # mint-teal
        "#F4CFB6",
        
        ]
    kb = 0.0083145
    T = 310
    kbT = kb * T
    sns.set_theme(style="ticks") # 清爽的刻度样式
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.size': 12,
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5
        })
    # apo_start_file = plumed.read_as_pandas(f"/workdir/AgOR10_28/AgOR10/AgOR10_apo/cuts1/COLVAR_alpha_angle_start")
    # apo_start = apo_start_file[f'{cv_name}'][0]
    # bound_start_file = plumed.read_as_pandas(f"/workdir/AgOR10_28/AgOR10/AgOR10_bound/COLVAR_alpha_angle_start")
    # bound_start = bound_start_file[f'{cv_name}'][0]
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, filename in enumerate(filenames):
        file_path = f"/workdir/AgOR10_28/AgOR10/PCCA2/FES/{filename}"
        index = indices[i]
        if not os.path.exists(file_path):
            continue
            
        data = plumed.read_as_pandas(file_path)
        
        # 1. 计算概率密度
        prob = np.exp(-data['file.free'] / kbT)
        prob = prob / np.sum(prob)*100 # 归一化()

        # 2. 排序以便绘图
        cv_values = data[cv_name].values
        if cv_name == 'angle':
            cv_values = cv_values/np.pi*180

        sort_idx = np.argsort(cv_values)
        ax.plot(cv_values[sort_idx], prob[sort_idx], 
                 label=f'{index}', color=colors[i], lw=2)
        # 可选：填充透明区域，增加视觉效果
        ax.fill_between(cv_values[sort_idx], prob[sort_idx], alpha=0.1, color=colors[i])
    if cv_name == 'angle':
        apo_start = apo_start/np.pi*180
        bound_start = bound_start/np.pi*180
    apo_start = 0.62
    bound_start = 0.97
    ax.axvline(x=apo_start, color='black', linestyle='--', linewidth=2, label='Apo-Closed Cryo-EM')
    ax.axvline(x=bound_start, color='red', linestyle='--', linewidth=2, label='Holo-Open Cryo-EM')
        # 4. 精修图表细节
    ax.set_xlabel('Minimum Pore Diameter (nm)', labelpad=10)
    # ax.set_xlabel(r'ALPHARMSD', labelpad=10)
    ax.set_ylabel('Probability Density (%)', labelpad=10)
    # ax.set_xlim(0, 0.5)
    # ax.set_ylim(0, 16)
    # 去掉上方和右方的边框
    sns.despine()

    # 添加网格（可选，淡淡的灰色网格有助于读数）
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # 优化图例
    ax.legend(frameon=False, loc='upper right', fontsize=11)

# 自动调整布局，防止边缘裁剪
    plt.tight_layout()
    plt.savefig(f'{cv_name}.pdf', dpi=300, transparent=True)
    plt.show()
plot_all_distributions('d1.lowest', ['FES_1_d1.lowest.dat','FES_2_d1.lowest.dat','FES_3_d1.lowest.dat','FES_4_d1.lowest.dat'], [
    r"$\mathrm{I'}$",
    r"$\mathrm{I}$",
    r"$\mathrm{O}$",
    r"$\mathrm{C}$",
])

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plumed


def plot_all_2distributions(
    cv_names,
    filenames,
    indices,
    output_file="2CV_distributions.pdf"
):

    kb = 0.0083145
    T = 310
    kbT = kb * T

    colors = [
        "#F6C667",  # soft gold
        "#B8A1E3",  # lavender
        "#00F56A",  # mint
        "#00B8D4",  # blue
    ]

    sns.set_theme(style="ticks")

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.size': 12,
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5
    })

    # ==================================================
    # read cryo-EM references
    # ==================================================
    apo_start_file = plumed.read_as_pandas(
        "/workdir/AgOR10_28/AgOR10/AgOR10_apo/cuts1/COLVAR_S6-S7_start"
    )

    bound_start_file = plumed.read_as_pandas(
        "/workdir/AgOR10_28/AgOR10/AgOR10_bound/COLVAR_S6-S7_start"
    )

    n_cv = len(cv_names)
    files_per_cv = len(indices)

    if len(filenames) != n_cv * files_per_cv:
        raise ValueError(
            f"Expected {n_cv * files_per_cv} files "
            f"but got {len(filenames)}."
        )

    # ==================================================
    # figure
    # ==================================================
    fig, axes = plt.subplots(
        n_cv,
        1,
        figsize=(7, 4),
        sharex=True,
        sharey=True
    )

    if n_cv == 1:
        axes = [axes]

    # ==================================================
    # loop over CVs
    # ==================================================
    for j, cv_name in enumerate(cv_names):

        ax = axes[j]

        apo_start = apo_start_file[cv_name].iloc[0]
        bound_start = bound_start_file[cv_name].iloc[0]

        cv_files = filenames[
            j * files_per_cv:
            (j + 1) * files_per_cv
        ]

        for i, filename in enumerate(cv_files):

            file_path = (
                f"/workdir/AgOR10_28/AgOR10/PCCA2/FES/{filename}"
            )

            if not os.path.exists(file_path):
                print(f"Missing file: {file_path}")
                continue

            data = plumed.read_as_pandas(file_path)

            prob = np.exp(
                -data["file.free"].values / kbT
            )

            prob = prob / np.sum(prob) * 100

            cv_values = data[cv_name].values

            if cv_name == "angle":
                cv_values = cv_values / np.pi * 180

            order = np.argsort(cv_values)

            ax.plot(
                cv_values[order],
                prob[order],
                lw=2,
                color=colors[i],
                label=indices[i]
            )

            ax.fill_between(
                cv_values[order],
                prob[order],
                color=colors[i],
                alpha=0.15
            )

        # ==========================================
        # Cryo-EM reference structures
        # ==========================================
        if cv_name == "angle":
            apo_start = apo_start / np.pi * 180
            bound_start = bound_start / np.pi * 180

        ax.axvline(
            apo_start,
            color="black",
            linestyle="--",
            linewidth=2,
            label="Apo-Closed Cryo-EM"
        )

        ax.axvline(
            bound_start,
            color="red",
            linestyle="--",
            linewidth=2,
            label="Holo-Open Cryo-EM"
        )

        # ==========================================
        # axis settings
        # ==========================================
        ax.set_xlim(0.18, 0.9)
        ax.set_ylim(0, 14)

        ax.set_xticks([0.2, 0.4, 0.6,0.8])
        ax.set_yticks([0, 4, 8, 12])

        ax.tick_params(
            axis='both',
            direction='out',
            length=5,
            width=1.5
        )


    # ==================================================
    # common labels
    # ==================================================
    fig.supylabel(
        "Probability Density (%)",
        fontsize=13
    )

    axes[-1].set_xlabel(
        "Distance (nm)",
        fontsize=13
    )

    sns.despine()

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        transparent=True,
        bbox_inches='tight'
    )

    plt.show()
plot_all_2distributions(
    cv_names=[
        "d_A_PHE356_A_ILE293.min",
        "d_A_PHE356_A_PHE294.min"
    ],

    filenames=[
        # CV1
        "FES_S6-S71_d_A_PHE356_A_ILE293.min.dat",
        "FES_S6-S72_d_A_PHE356_A_ILE293.min.dat",
        "FES_S6-S73_d_A_PHE356_A_ILE293.min.dat",
        "FES_S6-S74_d_A_PHE356_A_ILE293.min.dat",

        # CV2
        "FES_S6-S71_d_A_PHE356_A_PHE294.min.dat",
        "FES_S6-S72_d_A_PHE356_A_PHE294.min.dat",
        "FES_S6-S73_d_A_PHE356_A_PHE294.min.dat",
        "FES_S6-S74_d_A_PHE356_A_PHE294.min.dat",
    ],

    indices=[
        "Off-path Intermediate",
        "Intermediate",
        "Open",
        "Closed"
    ]
)
